In [ ]:
# kaggle API를 사용하기 위해 API 키 json 파일 업로드

from google.colab import files
files.upload()


Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"jhw0714","key":"5ebfcd63c0dd4e608eb3f7b6fbbb5ebd"}'}

In [ ]:
# kaggle.json 파일을 적절한 디렉토리로 옮긴다.
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json


# 데이터셋을 다운로드한다.
!kaggle datasets download -d vladimirvorobevv/chatgpt-paraphrases
!unzip -o /content/chatgpt-paraphrases.zip -d /content/sample_data

 65% 39.0M/60.0M [00:00<00:00, 144MB/s]
100% 60.0M/60.0M [00:00<00:00, 157MB/s]
Archive:  /content/chatgpt-paraphrases.zip
  inflating: /content/sample_data/chatgpt_paraphrases.csv  


In [ ]:
# 파이스파크 설치
!pip install pyspark
!pip install -U -q PyDrive
!apt install openjdk-8-jdk-headless -qq

# 한국어 텍스트 전처리를 도와주는 NLTK 패키지
!pip install nltk

import os
import re
from google.colab import drive
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
drive.mount('/content/drive')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.9/316.9 MB 3.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pyspark: filename=pyspark-3.5.0-py2.py3-none-any.whl size=317425344 sha256=787daa87065028cae23baaef5cbba9782c8e0f79f14899c032e96be45e9e8839
  Stored in directory: /root/.cache/pip/wheels/41/4e/10/c2cf2467f71c678cfc8a6b9ac9241e5e44a01940da8fbb17fc
Successfully built pyspark
The following additional packages will be installed:
  libxtst6 openjdk-8-jre-headless
Suggested packages:
  openjdk-8-demo openjdk-8-source libnss-mdns fonts-dejavu-extra fonts-nanum fonts-ipafont-gothic
  fonts-ipafont-mincho fonts-wqy-microhei fonts-wqy-zenhei fonts-indic
The following NEW packages will be installed:
  libxtst6 openjdk-8-jdk-headless openjdk-8-jre-headless
0 upgraded, 3 newly installed, 0 to remove and 10 not upgraded.
Need to get 39.7 MB of archives.
After this operation, 144 MB of additional disk space will be used.
Selecting previously unselected package

In [ ]:
# Let's import the libraries we will need
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

import pyspark
from pyspark.sql import *
from pyspark.sql.functions import *
# SparkContext is the entry point to any spark functionality.
# SparkConf provides configurations to run a Spark application.
from pyspark import SparkContext, SparkConf

# create the Spark Session
spark = SparkSession.builder.appName("WordCount").getOrCreate()

# create the Spark Context
sc = spark.sparkContext

In [ ]:
data=pd.read_csv('/content/sample_data/chatgpt_paraphrases.csv')

In [ ]:
# 각 문장을 사람이 작성했는지, AI가 작성했는지 태그
category={}
for i in range(len(data)):
    chatgpt=data.iloc[i]["paraphrases"][1:-1].split(', ')
    for j in chatgpt[:1]:
        category[j[1:-1]]='chatgpt'
    category[data.iloc[i]['text']]="human"

In [ ]:
# 사람의 작성한 문장과 AI가 작성한 문장을 분리
totla_paraphrases=pd.DataFrame(category.items(),columns=["text","category"])
human_paraphrases = totla_paraphrases[totla_paraphrases['category'] == 'human']
gpt_paraphrases = totla_paraphrases[totla_paraphrases['category'] == 'chatgpt']

In [ ]:
import nltk

from nltk.stem import PorterStemmer
from nltk.stem import LancasterStemmer
from nltk.stem import RegexpStemmer
from nltk.stem import WordNetLemmatizer

from nltk.tokenize import sent_tokenize
from nltk.tokenize import word_tokenize

from nltk.corpus import stopwords

nltk.download('punkt')
nltk.download('wordnet')
nltk.download('stopwords')

lemmatizer = WordNetLemmatizer()

use_stopwords = False
english_stop_list = [".", ",", "!", "?", ")", "(", "'", '"', "]", "[", "{", "}", ":", ";", "<", ">", "~", "''", "``"]

if use_stopwords:
  english_stop_list += list(stopwords.words('english'))

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


# 패턴 찾아내기

In [ ]:
from pyspark.ml.fpm import FPGrowth

In [ ]:
human_wordbasket=[]
# len(human_paraphrases)
for i in range(400000):
  listofword = word_tokenize(human_paraphrases.iloc[i]['text'].lower())
  for w in listofword:
    if w not in english_stop_list:
      human_wordbasket.append((i, lemmatizer.lemmatize(w , pos="v")))

In [ ]:
human_worddata = pd.DataFrame(human_wordbasket, columns=['paraphrases_id', 'word'])
human_worddata_spark = spark.createDataFrame(human_worddata)
human_wordbag  = human_worddata_spark.groupBy("paraphrases_id").agg(collect_set('word').alias('words'))

In [ ]:
human_wordbag.show(20)

+--------------+--------------------+
|paraphrases_id|               words|
+--------------+--------------------+
|             0|[the, in, market,...|
|             7|[be, geologist, i...|
|            19|[rocket, white, w...|
|            22|[much, be, 30, hp...|
|            26|[much, society, i...|
|            29|['s, one, better,...|
|            34|[obama, their, tr...|
|            50|[player, blu, so,...|
|            54|[it, rsi, be, get...|
|            57|[the, off, email,...|
|            65|[the, be, best, b...|
|            77|[the, through, mo...|
|            94|[3d, work, how, d...|
|           110|[the, be, world, ...|
|           112|[china, do, think...|
|           113|[mean, of, do, so...|
|           126|[excessive, amoun...|
|           130|[talk, party, be,...|
|           136|[be, endocytosis,...|
|           144|[fab, rsus, offer...|
+--------------+--------------------+
only showing top 20 rows



In [ ]:
human_fpGrowth = FPGrowth(itemsCol="words", minSupport=0.01, minConfidence=0.5)
human_model = human_fpGrowth.fit(human_wordbag)

In [ ]:
print("{} frequent itemsets were generated.\n=================================================================".format(human_model.freqItemsets.count()))
freqSets = human_model.freqItemsets.sort("freq", ascending=False)
freqSets.show(20, truncate=False)

print("{} association rules were generated.\n=================================================================".format(human_model.associationRules.count()))
AssoRules = human_model.associationRules.sort("support", ascending=False)
AssoRules.show(20, truncate=False)

1437 frequent itemsets were generated.
+----------+------+
|items     |freq  |
+----------+------+
|[be]      |224305|
|[the]     |209722|
|[the, be] |134709|
|[of]      |122218|
|[in]      |121553|
|[to]      |117170|
|[a]       |111630|
|[and]     |108578|
|[what]    |99866 |
|[of, the] |96248 |
|[in, the] |80991 |
|[what, be]|76978 |
|[do]      |75937 |
|[of, be]  |75482 |
|[and, the]|75151 |
|[in, be]  |72235 |
|[to, the] |71973 |
|[to, be]  |71243 |
|[and, be] |65132 |
|[a, be]   |62803 |
+----------+------+
only showing top 20 rows

1285 association rules were generated.
+-----------+----------+------------------+------------------+-------------------+
|antecedent |consequent|confidence        |lift              |support            |
+-----------+----------+------------------+------------------+-------------------+
|[be]       |[the]     |0.6005617351374245|1.1454348726874835|0.3367750258126936 |
|[the]      |[be]      |0.6423217402084664|1.1454348726874835|0.3367750258126936 |
|

In [ ]:
rules_generated = AssoRules.join(freqSets, AssoRules.consequent == freqSets.items)\
.withColumn("interest",abs(col("confidence") - (col("freq")/human_wordbag.count())))\
.select('antecedent', 'consequent', 'confidence', 'support', 'interest').sort("interest", ascending=False, truncate=False)

In [ ]:
freqSets.toPandas()

,items,freq
0,[be],224305
1,[the],209722
2,"[the, be]",134709
3,[of],122218
4,[in],121553
...,...,...
1432,"[my, and]",4007
1433,"[that, and, in, the, be]",4006
1434,"['s, with]",4005
1435,"[from, a, in]",4004


In [ ]:
rules_generated.toPandas()

,antecedent,consequent,confidence,support,interest
0,[difference],[between],0.929558,0.010260,0.904201
1,"[my, how]",[i],0.957148,0.018763,0.821910
2,"[my, can]",[i],0.937552,0.014113,0.802313
3,"[my, do]",[i],0.879423,0.015845,0.744184
4,"[between, what, the]",[and],0.953555,0.011600,0.682108
...,...,...,...,...,...
1280,"['s, to, the]",[be],0.559056,0.013088,0.001711
1281,[a],[the],0.523058,0.145974,0.001251
1282,"[have, of]",[be],0.559935,0.025680,0.000832
1283,"['s, and, the]",[be],0.561428,0.013995,0.000661
